# Early Warning Prediction

This notebook uses the final XGBoost model to generate early warning predictions for future air quality deterioration.

The trained model produces a probability of deterioration for each observation. These probabilities are then converted into early warning signals using a selected decision threshold.

The analysis focuses on evaluating the model as an early warning system rather than training a new model.


## 1. Load Final Model

In [1]:
from pathlib import Path
import joblib
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier


In [2]:
MODEL_PATH = Path("../models/xgboost_final.joblib")

final_xgb = joblib.load(MODEL_PATH)

print("Final XGBoost model loaded successfully.")
print(f"Model path: {MODEL_PATH}")

Final XGBoost model loaded successfully.
Model path: ..\models\xgboost_final.joblib


### Model Fitting Verification

The loaded XGBoost model is checked to confirm that the estimator was fitted successfully before generating early warning predictions.


In [3]:
is_fitted = hasattr(
    final_xgb.named_steps["model"],
    "n_features_in_"
)

print(f"XGBoost fitted: {is_fitted}")

XGBoost fitted: True


## 2. Prepare Test Data

The unseen test set used for final model evaluation is prepared for early warning prediction.

The test period covers January 2016 to February 2017 and was not used during model training or hyperparameter tuning.

The same test period is retained to ensure consistency with the final model evaluation.


In [4]:
DATA_PATH = Path("../data/processed/airshift_labeled.csv")

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["datetime"]
)

TEST_START = "2016-01-01"
TEST_END = "2017-02-28 17:00:00"

test_data = df[
    (df["datetime"] >= TEST_START) &
    (df["datetime"] <= TEST_END)
].copy()

print("Test data shape:", test_data.shape)
print("Test date range:")
print(test_data["datetime"].min())
print(test_data["datetime"].max())

Test data shape: (121900, 100)
Test date range:
2016-01-01 00:00:00
2017-02-28 17:00:00


## 3. Define Prediction Features

The target variable is separated from the input features before generating early warning predictions.

The `Deterioration` column is retained as the actual outcome for evaluating the generated warnings, while `No` and `datetime` are excluded because they were not used as model features during training.


In [5]:
TARGET = "Deterioration"

EXCLUDE_COLUMNS = [
    TARGET,
    "No",
    "datetime"
]

X_test = test_data.drop(columns=EXCLUDE_COLUMNS)
y_test = test_data[TARGET]

print("Test features shape:", X_test.shape)
print("Test target shape:", y_test.shape)
print("Number of features:", X_test.shape[1])

Test features shape: (121900, 97)
Test target shape: (121900,)
Number of features: 97


## 4. Generate Deterioration Probabilities

The final XGBoost model is used to generate the predicted probability of future air quality deterioration for each observation in the unseen test set.

The probability for the positive class (`Deterioration = 1`) is retained for subsequent conversion into early warning signals using a decision threshold.


In [6]:
y_test_prob = final_xgb.predict_proba(X_test)[:, 1]

print("Predicted probabilities generated successfully.")
print("Number of predictions:", len(y_test_prob))
print("Minimum probability:", y_test_prob.min())
print("Maximum probability:", y_test_prob.max())
print("Mean probability:", y_test_prob.mean())

Predicted probabilities generated successfully.
Number of predictions: 121900
Minimum probability: 0.0004506121
Maximum probability: 0.9995968
Mean probability: 0.4957185


## 5. Warning Threshold

The predicted deterioration probabilities are converted into early warning signals using a decision threshold.

An observation is classified as an early warning when its predicted probability is equal to or greater than the selected threshold.

Because the system is designed for early warning, the threshold should be evaluated based on the trade-off between detecting deterioration events and generating false alarms.


In [7]:
thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

threshold_results = []

for threshold in thresholds:
    y_warning = (y_test_prob >= threshold).astype(int)

    precision = precision_score(y_test, y_warning)
    recall = recall_score(y_test, y_warning)
    f1 = f1_score(y_test, y_warning)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_warning
    ).ravel()

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "False Positives": fp,
        "False Negatives": fn
    })

threshold_results_df = pd.DataFrame(threshold_results)

threshold_results_df

,Threshold,Precision,Recall,F1,False Positives,False Negatives
0,0.30,0.625185,0.899818,0.737772,31652,5878
1,0.35,0.649035,0.865372,0.741751,27456,7899
2,0.40,0.673316,0.825542,0.741699,23501,10236
3,0.45,0.699033,0.781467,0.737955,19741,12822
4,0.50,0.724362,0.731461,0.727894,16331,15756
5,0.55,0.750531,0.680125,0.713596,13264,18768
6,0.60,0.776531,0.625177,0.692682,10556,21992
7,0.65,0.804246,0.568166,0.665901,8114,25337
8,0.70,0.831526,0.506996,0.629919,6027,28926


In [8]:
MODEL_DIR = Path("../models")

print("Saved model files:")
for file in MODEL_DIR.iterdir():
    print("-", file.name)

Saved model files:
- xgboost_final.joblib


In [9]:
TRAIN_END = "2014-12-31 23:00:00"
VALIDATION_START = "2015-01-01"
VALIDATION_END = "2015-12-31 23:00:00"

development_data = df[
    df["datetime"] <= TRAIN_END
].copy()

validation_data = df[
    (df["datetime"] >= VALIDATION_START) &
    (df["datetime"] <= VALIDATION_END)
].copy()

print("Development data shape:", development_data.shape)
print("Development date range:")
print(development_data["datetime"].min())
print(development_data["datetime"].max())

print("\nValidation data shape:", validation_data.shape)
print("Validation date range:")
print(validation_data["datetime"].min())
print(validation_data["datetime"].max())

Development data shape: (191665, 100)
Development date range:
2013-03-01 00:00:00
2014-12-31 23:00:00

Validation data shape: (104816, 100)
Validation date range:
2015-01-01 00:00:00
2015-12-31 23:00:00


### 1. Prepare Features and Target

The target variable is separated from the input features for the development and validation periods.

The same feature exclusions used during model development are applied to maintain consistency with the final XGBoost model.


In [10]:
TARGET = "Deterioration"

EXCLUDE_COLUMNS = [
    TARGET,
    "No",
    "datetime"
]

X_development = development_data.drop(columns=EXCLUDE_COLUMNS)
y_development = development_data[TARGET]

X_validation = validation_data.drop(columns=EXCLUDE_COLUMNS)
y_validation = validation_data[TARGET]

print("Development features shape:", X_development.shape)
print("Development target shape:", y_development.shape)

print("\nValidation features shape:", X_validation.shape)
print("Validation target shape:", y_validation.shape)

Development features shape: (191665, 97)
Development target shape: (191665,)

Validation features shape: (104816, 97)
Validation target shape: (104816,)


### 5.2 Train Threshold Selection Model

A separate XGBoost model is fitted using only the 2013–2014 development period.

The model uses the same tuned hyperparameters selected during model development. The 2015 validation period is then used exclusively to evaluate different warning thresholds.

This prevents the final test set from influencing the threshold selection process.


In [11]:
print("Final model pipeline steps:")
print(final_xgb.named_steps.keys())

print("\nPreprocessor:")
print(final_xgb.named_steps["preprocessor"])

Final model pipeline steps:
dict_keys(['preprocessor', 'model'])

Preprocessor:
ColumnTransformer(transformers=[('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['wd', 'station']),
                                ('numerical', SimpleImputer(strategy='median'),
                                 ['year', 'month', 'day', 'hour', 'PM2.5',
                                  'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP',
                                  'PRES', 'DEWP', 'RAIN', 'WSPM', 'day_of_week',
                                  'is_weekend', 'PM2.5_lag_1h', 'PM2.5_lag_3h',
                                  'PM2.5_lag_6h', 'PM10_lag_1h', 'PM10_lag_3h',
                                  'PM10_

In [12]:

threshold_preprocessor = clone(
    final_xgb.named_steps["preprocessor"]
)

threshold_model = Pipeline(
    steps=[
        ("preprocessor", threshold_preprocessor),
        (
            "model",
            XGBClassifier(
                n_estimators=400,
                learning_rate=0.1,
                max_depth=6,
                min_child_weight=3,
                subsample=0.7,
                colsample_bytree=1.0,
                random_state=42,
                n_jobs=2,
                eval_metric="logloss"
            )
        )
    ]
)

print("Starting threshold-selection model training...")
print("Development data:", X_development.shape)

threshold_model.fit(
    X_development,
    y_development
)

print("Threshold-selection model training completed successfully.")

Starting threshold-selection model training...
Development data: (191665, 97)
Threshold-selection model training completed successfully.


### 5.3 Generate Validation Probabilities

The threshold-selection model is used to generate deterioration probabilities for the unseen 2015 validation period.

These probabilities will be used to evaluate different warning thresholds before the selected threshold is applied to the final test period.


In [13]:
y_validation_prob = threshold_model.predict_proba(
    X_validation
)[:, 1]

print("Validation probabilities generated successfully.")
print("Number of predictions:", len(y_validation_prob))
print("Minimum probability:", y_validation_prob.min())
print("Maximum probability:", y_validation_prob.max())
print("Mean probability:", y_validation_prob.mean())

Validation probabilities generated successfully.
Number of predictions: 104816
Minimum probability: 9.9503086e-05
Maximum probability: 0.9992524
Mean probability: 0.48155317


### 5.4 Select Warning Threshold

Different decision thresholds are evaluated on the 2015 validation period to examine the trade-off between precision and recall.

The validation period is used only for selecting the operating threshold. The final test period remains untouched until the selected threshold is applied to the final XGBoost model.


In [14]:
thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

validation_threshold_results = []

for threshold in thresholds:
    y_warning = (y_validation_prob >= threshold).astype(int)

    precision = precision_score(y_validation, y_warning)
    recall = recall_score(y_validation, y_warning)
    f1 = f1_score(y_validation, y_warning)

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        y_warning
    ).ravel()

    validation_threshold_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "False Positives": fp,
        "False Negatives": fn
    })

validation_threshold_results_df = pd.DataFrame(
    validation_threshold_results
)

validation_threshold_results_df

,Threshold,Precision,Recall,F1,False Positives,False Negatives
0,0.30,0.620139,0.881260,0.727993,26795,5894
1,0.35,0.642142,0.845219,0.729817,23381,7683
2,0.40,0.664963,0.804565,0.728133,20122,9701
3,0.45,0.687440,0.760244,0.722011,17158,11901
4,0.50,0.711215,0.710585,0.710900,14322,14366
5,0.55,0.736852,0.659616,0.696098,11693,16896
6,0.60,0.761737,0.604073,0.673805,9379,19653
7,0.65,0.788024,0.547464,0.646078,7310,22463
8,0.70,0.816885,0.488094,0.611070,5431,25410


### 5.5 Selected Warning Threshold

A threshold of **0.30** is selected for the early warning system based on the 2015 validation results.

The threshold is chosen with emphasis on **recall**, since missing an actual deterioration event is particularly important in an early warning context.

At this threshold, the validation results are:

* Precision: **0.6201**
* Recall: **0.8813**
* F1-score: **0.7280**
* False negatives: **5,894**

The selected threshold is determined using the validation period only. The final test period remains unseen and is used afterward to evaluate the complete early warning system.


## 6. Final Early Warning Performance

The selected warning threshold of **0.30** is applied to the deterioration probabilities generated by the final XGBoost model on the unseen 2016–2017 test period.

This provides the final performance evaluation of the AirShift early warning system. The test set is used only at this stage and was not used to select the model or warning threshold.


In [15]:
WARNING_THRESHOLD = 0.30

y_warning = (
    y_test_prob >= WARNING_THRESHOLD
).astype(int)

precision = precision_score(y_test, y_warning)
recall = recall_score(y_test, y_warning)
f1 = f1_score(y_test, y_warning)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    y_warning
).ravel()

print("Final Early Warning Performance")
print("-" * 40)
print(f"Warning threshold: {WARNING_THRESHOLD:.2f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"True Negatives: {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives: {tp}")

Final Early Warning Performance
----------------------------------------
Warning threshold: 0.30
Precision: 0.6252
Recall: 0.8998
F1-score: 0.7378
True Negatives: 31575
False Positives: 31652
False Negatives: 5878
True Positives: 52795
